## Purpose
This notebook builds an **inhaled asthma medication list** by linking **ATC R03 classes** to **FDA NDC package codes** and then joining to the FDA product directory to keep only **inhalation-route** products.

## High-level workflow
- **Input 1**: `package_NDC_ATC4_classes.csv`
  - Contains mappings between an `NDC` identifier and an `ATC_class`.
  - We filter to rows where `ATC_class` contains **`R03`** (Drugs for obstructive airway diseases).
- **Input 2**: `package.txt` (tab-separated)
  - Used to map **`NDCPACKAGECODE` → PRODUCTID**.
- **Input 3**: `product.xlsx`
  - FDA product directory export.
  - Used to retrieve product attributes (including `ROUTENAME`, `SUBSTANCENAME`, etc.).

## Step-by-step logic (what the code does)
- **(1) ATC → NDC**: extract NDCs associated with ATC class `R03`.
- **(2) NDC → PRODUCTID**: use the package file to obtain a list of `PRODUCTID` values for those NDCs.
- **(3) PRODUCTID → Product metadata**: subset the product table to those `PRODUCTID`s.
- **(4) Route filter**: keep only products where `ROUTENAME` contains **`INHALATION`**.
- **(5) Review surface**: export an intermediate file (`inhaler.csv`) and inspect `SUBSTANCENAME` values.
- **(6) Manual classification**: map `SUBSTANCENAME` to an asthma medication class via `class_map`.
  - This is a **manual curation step**. If a substance is not an asthma medication, the class will be blank.
- **(7) Attach package NDC**: merge back the package-level code (`NDCPACKAGECODE`).
- **(8) Normalize to 11 digits**: convert hyphenated package NDCs into **11-digit** NDC (5-4-2) as text.
- **(9) Export**: write `additional_inhalers.xlsx`.

## Output
- **`additional_inhalers.xlsx`**
  - `NDC`: 11-digit NDC (string)
  - `Generic name`: derived from `SUBSTANCENAME` (the ingredient / ingredient-combination string)
  - `asthma_med_class_new`: the manually assigned class (e.g., ICS/LABA/LAMA)

## Notes / caveats
- This notebook identifies inhalation-route products linked to ATC R03. **Not everything returned is necessarily indicated for asthma** (some are COPD-only, and indications vary by country and label).
- The classification step requires **Physician's manual review**.


In [7]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# Goal
# ---------------------------------------------------------
# Build an inhaler-focused medication list by linking:
#   ATC R03 classes  →  NDC package codes  →  FDA PRODUCTID  →  product metadata
# and then filtering to products whose route includes "INHALATION".
#
# Why this approach?
# - ATC provides a high-level therapeutic grouping (R03 = obstructive airway diseases).
# - NDC package codes are the practical identifiers often used in US datasets.
# - FDA product metadata (ROUTENAME, SUBSTANCENAME) helps us narrow to true inhalers.
#
# IMPORTANT CAVEAT
# - R03 includes asthma and COPD medications; this pipeline is intentionally inclusive.
# - The final "asthma_med_class_new" column is a *manual curation* step and may require review.

# ---------------------------------------------------------
# Inputs
# ---------------------------------------------------------
# package_NDC_ATC4_classes.csv
#   - contains columns like: NDC, ATC_class
#   - built upstream (e.g., via n2c mappings)
#
# package.txt (tab-separated FDA package file)
#   - contains: NDCPACKAGECODE, PRODUCTID, etc.
#
# product.xlsx (FDA product file export)
#   - contains: PRODUCTID, ROUTENAME, SUBSTANCENAME, etc.
#
# NOTE: All file paths below are relative to the current working directory.
package_NDC_ATC_classes = pd.read_csv("package_NDC_ATC4_classes.csv")
package = pd.read_csv("package.txt", sep="\t")
product = pd.read_excel("product.xlsx")

# ---------------------------------------------------------
# Step 1: ATC → NDC
# ---------------------------------------------------------
# Extract NDC package codes linked to ATC class R03.
# We use string contains("R03") so we capture all R03 sub-classes.
# (Example: R03AC, R03AK, etc.)
inhaler_list = (
    package_NDC_ATC_classes
    .loc[
        package_NDC_ATC_classes["ATC_class"].str.contains("R03", na=False),
        "NDC"
    ]
    .drop_duplicates()
    .tolist()
)

# ---------------------------------------------------------
# Step 2: NDC → PRODUCTID
# ---------------------------------------------------------
# Map NDCPACKAGECODE → PRODUCTID using the FDA package file.
# This gives us the set of product records corresponding to ATC R03-linked NDCs.
inhaler_product = (
    package
    .loc[
        package["NDCPACKAGECODE"].isin(inhaler_list),
        "PRODUCTID"
    ]
    .drop_duplicates()
    .tolist()
)

# ---------------------------------------------------------
# Step 3: PRODUCTID → product metadata
# ---------------------------------------------------------
# Subset the FDA product table to only PRODUCTIDs in our ATC-derived list.
inhaler = product[
    product["PRODUCTID"].isin(inhaler_product)
].copy()

# ---------------------------------------------------------
# Step 4: Route filter (INHALATION)
# ---------------------------------------------------------
# Narrow to products where ROUTENAME includes "INHALATION".
# This is a broad but practical way to focus on inhaler/nebulized products.
inhaler = inhaler[
    inhaler["ROUTENAME"].str.contains("INHALATION", na=False)
].copy()

# ---------------------------------------------------------
# Step 5: Quick review surface
# ---------------------------------------------------------
# SUBSTANCENAME is the key ingredient string used for manual classification.
# Inspecting unique values helps you refine class_map below.
inhaler["SUBSTANCENAME"].drop_duplicates().sort_values()

# Export an intermediate table for manual review / auditing.
# This is useful when you want to inspect the product list in Excel.
inhaler.to_csv("inhaler.csv", index=False)

# ---------------------------------------------------------
# Step 6: Manual asthma/COPD class assignment
# ---------------------------------------------------------
# class_map maps FDA SUBSTANCENAME strings to a simplified class label.
# Examples:
# - ICS: inhaled corticosteroid
# - LABA: long-acting beta-agonist
# - LAMA: long-acting muscarinic antagonist
# - ICS/LABA/LAMA: triple therapy
#
# NOTE:
# - SUBSTANCENAME strings must match exactly.
# - If a SUBSTANCENAME is missing from this dictionary, its class will be NaN.
# - This mapping is intentionally editable; update as you add/remove substances.
class_map = {
    "ACLIDINIUM BROMIDE": "LAMA",
    "ACLIDINIUM BROMIDE; FORMOTEROL FUMARATE": "LABA/LAMA",
    "ALBUTEROL SULFATE": "SABA",
    "ALBUTEROL SULFATE; BUDESONIDE": "SABA/ICS",
    "ALBUTEROL SULFATE; IPRATROPIUM BROMIDE": "SABA/SAMA",
    "BECLOMETHASONE DIPROPIONATE": "ICS",
    "BUDESONIDE": "ICS",
    "BUDESONIDE; FORMOTEROL FUMARATE": "ICS/LABA",
    "BUDESONIDE; FORMOTEROL; GLYCOPYRROLATE": "ICS/LABA/LAMA",
    "CICLESONIDE": "ICS",
    "CROMOLYN SODIUM": "Cromolyn",
    "EPINEPHRINE": "Epinephrine",
    "FLUTICASONE FUROATE": "ICS",
    "FLUTICASONE FUROATE; UMECLIDINIUM BROMIDE; VILANTEROL TRIFENATATE": "ICS/LABA/LAMA",
    "FLUTICASONE FUROATE; VILANTEROL TRIFENATATE": "ICS/LABA",
    "FLUTICASONE PROPIONATE": "ICS",
    "FLUTICASONE PROPIONATE; SALMETEROL XINAFOATE": "ICS/LABA",
    "FORMOTEROL FUMARATE": "LABA",
    "FORMOTEROL FUMARATE; GLYCOPYRROLATE": "LABA/LAMA",
    "FORMOTEROL FUMARATE; MOMETASONE FUROATE": "ICS/LABA",
    "IPRATROPIUM BROMIDE": "LAMA",
    "LEVALBUTEROL": "SABA",
    "LEVALBUTEROL HYDROCHLORIDE": "SABA",
    "MOMETASONE FUROATE": "ICS",
    "OLODATEROL HYDROCHLORIDE": "LABA",
    "OLODATEROL HYDROCHLORIDE; TIOTROPIUM BROMIDE MONOHYDRATE": "LABA/LAMA",
    "REVEFENACIN": "LAMA",
    "SALMETEROL XINAFOATE": "LABA",
    "TIOTROPIUM BROMIDE ANHYDROUS": "LAMA",
    "TIOTROPIUM BROMIDE MONOHYDRATE": "LAMA",
    "UMECLIDINIUM BROMIDE; VILANTEROL TRIFENATATE": "LABA/LAMA"
}

# Assign class labels onto the inhaler table.
inhaler["asthma_med_class_new"] = inhaler["SUBSTANCENAME"].map(class_map)

# ---------------------------------------------------------
# Step 7: Attach NDCPACKAGECODE back to the inhaler table
# ---------------------------------------------------------
# We re-join PRODUCTID to NDCPACKAGECODE because PRODUCTID is the key that connects
# product metadata to package-level NDC identifiers.
inhaler_PRODUCTID = inhaler["PRODUCTID"].drop_duplicates().tolist()

inhaler_product_ndc = package[
    package["PRODUCTID"].isin(inhaler_PRODUCTID)
][["PRODUCTID", "NDCPACKAGECODE"]]

final = inhaler.merge(inhaler_product_ndc, on="PRODUCTID", how="left")

# ---------------------------------------------------------
# Step 8: Convert package NDC to digits-only 11-digit string
# ---------------------------------------------------------
# FDA package codes are typically hyphenated (labeler-product-package).
# We zero-pad each segment to the FDA 5-4-2 standard and then concatenate.
#
# Output is digits-only (no hyphens) to make joins/deduplication easier downstream.
def convert_ndc_to_11(ndc):
    if pd.isna(ndc):
        return np.nan

    parts = ndc.split("-")
    if len(parts) != 3:
        # If the input is not hyphenated into 3 parts, we do not attempt to guess.
        return np.nan

    part1 = parts[0].zfill(5)
    part2 = parts[1].zfill(4)
    part3 = parts[2].zfill(2)

    return part1 + part2 + part3

final["NDC11"] = final["NDCPACKAGECODE"].apply(convert_ndc_to_11)

# ---------------------------------------------------------
# Step 9: Final selection and export
# ---------------------------------------------------------
# Keep only the columns needed for downstream workflows.
# - NDC: 11-digit digits-only
# - Generic name: FDA substance string (ingredient / ingredient-combination)
# - asthma_med_class_new: manual classification label
cols_out = ["NDC11", "SUBSTANCENAME", "asthma_med_class_new"]
for c in cols_out:
    if c not in final.columns:
        final[c] = np.nan

final = final[cols_out].rename(columns={
    "NDC11": "NDC",
    "SUBSTANCENAME": "Generic name",
})

# Export to Excel for easy review.
final.to_excel("additional_inhalers.xlsx", index=False)